# Stochastic inventory control by Q-learning

The inventory problem of Unit 2, solved again -- this time without the
model. A shop holds a stock $i\in\{0,\dots,N\}$ of an item over $h$ stages.
At each stage it orders $j$ more (up to the capacity $N-i$), pays an
ordering cost and a holding cost, and sells whatever the demand takes. The
demand is $d=[2,2,1,1]$ at stages $1\dots4$, except that at stage $3$ it is
$2$ with probability $\tfrac12$ and $0$ with probability $\tfrac12$. At the
end the leftover stock is worth a salvage value.

Costs are negative profits, so minimising is maximising revenue.

The state is the pair (stock, stage), flattened to a single index -- which
is the state augmentation of Unit 10 that turns a finite-horizon problem
into a shortest path problem. The states at the final stage are terminal.
With that done, the stationary `q_learning_ssp` driver applies unchanged.

The exact answer is available by dynamic programming, so at the end we can
see precisely where Q-learning got it right and where it did not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from q_learning import q_learning_ssp

rng = np.random.default_rng(0)

In [ ]:
N, h = 4, 4                 # max stock, number of stages
n1   = N + 1                # stock levels 0..N

def ind2grid(x):            # flat index -> (stock, stage)
    return x % n1, x // n1

def grid2ind(i, k):         # (stock, stage) -> flat index
    return i + k * n1

price, c_item, c_tr, c_hold = 10, 5, 0.5, 0.2
demand = [2, 2, 1, 1]
salvage = -np.array([0, c_item - c_hold, 2 * (c_item - c_hold),
                     3 * (c_item - c_hold), 4 * (c_item - c_hold)])

def model_and_cost(x, u):
    """Simulator: order u units in state x, return the next state and cost."""
    i, k = ind2grid(x)
    if k == h:                                   # final stage: collect salvage
        return x, salvage[i]
    j  = u                                       # units ordered
    c2 = c_item * j + (c_tr if j > 0 else 0)     # ordering + fixed transport cost
    c1 = i * c_hold                              # holding cost
    if k != 2:
        i_next = max(i + j - demand[k], 0)
        cost   = -(min(demand[k], j + i) * price - (c1 + c2))
    else:                                        # stage 3: demand is random
        if rng.random() > 0.5:
            i_next = max(i + j - demand[k], 0)
            cost   = -(min(demand[k], j + i) * price - (c1 + c2))
        else:
            i_next = max(i + j, 0)
            cost   = -(min(0, j + i) * price - (c1 + c2))
    return grid2ind(i_next, k + 1), cost


# one action per feasible order quantity; terminal states have a single one
num_actions = np.array([1 if ind2grid(x)[1] == h else N + 1 - ind2grid(x)[0]
                        for x in range(n1 * (h + 1))])
terminal    = list(range(n1 * h, n1 * (h + 1)))
print(f"{n1 * (h + 1)} states, terminal states {terminal}")

In [ ]:
Q, mu, J = q_learning_ssp(
    num_actions, model_and_cost,
    nsim=300_000, eps=0.8, alpha=0.1,
    prob_init_state=np.r_[np.ones(n1) / n1, np.zeros(n1 * h)],
    terminal_states=terminal, rng=rng)

J_learned  = np.array([[J[grid2ind(i, k)]  for k in range(h + 1)] for i in range(n1)])
mu_learned = np.array([[mu[grid2ind(i, k)] for k in range(h + 1)] for i in range(n1)])
print("learned J (rows = stock 0..N, columns = stage 0..h):")
print(np.round(J_learned, 2))

In [ ]:
def exact_dp():
    """The same problem solved with the model, for comparison."""
    J = np.zeros((N + 1, h + 1)); J[:, h] = salvage
    mu = np.zeros((N + 1, h), dtype=int)
    for k in range(h - 1, -1, -1):
        for i in range(N + 1):
            best, best_u = np.inf, 0
            for j in range(N + 1 - i):
                c2 = c_item * j + (c_tr if j > 0 else 0)
                c1 = i * c_hold
                if k != 2:
                    i2  = max(i + j - demand[k], 0)
                    val = -(min(demand[k], j + i) * price - (c1 + c2)) + J[i2, k + 1]
                else:
                    iA, iB = max(i + j - demand[k], 0), max(i + j, 0)
                    cA = -(min(demand[k], j + i) * price - (c1 + c2))
                    cB = -(min(0, j + i) * price - (c1 + c2))
                    val = 0.5 * (cA + J[iA, k + 1]) + 0.5 * (cB + J[iB, k + 1])
                if val < best:
                    best, best_u = val, j
            J[i, k], mu[i, k] = best, best_u
    return J, mu


J_exact, mu_exact = exact_dp()
print("exact J:");            print(np.round(J_exact, 2))
print("\nerror (learned - exact):"); print(np.round(J_learned - J_exact, 2))

Read the error table by column.

**Stage 0** is accurate to a few hundredths everywhere: every simulation
starts there, with the stock drawn uniformly, so all five states are
visited constantly.

**Stage 3** is exact. Its Q-factors are set directly from the salvage
value, with no lookahead -- this is Step 1 of the convergence proof in the
unit, and it converges first and fastest.

**Stages 1 and 2 at high stock** are simply wrong: the entries are still
at their initial value of zero. Those states are never reached. The
trajectories start at stage 0 and order down, so a stock of 3 or 4 at
stage 1 would require ordering heavily into an already full shop, which
the learned policy never does -- and after the first thousand simulations
the exploration rate drops to `eps/300`, so chance never does it either.

That is the first condition of the convergence theorem failing in front of
us: *all state-action pairs must be visited infinitely often*. Q-learning
is not converging slowly on those states; it is not learning them at all.
Raising `eps`, or starting simulations from states other than stage 0,
fixes it.